# Matching road segments and grouping roads

For each crash, find out whether its road went from 50 to 30 km/h in December 2023.

**Sources**
- `ongevallen_gefilterd_amsterdam.csv`: crashes in Amsterdam 2016-2025, with `WVK_ID` (road segment) or `JTE_ID` (junction)
- `data/amsterdam_dertig.json`: road segments with old and new road type (`WEGTYPE_OUD`/`WEGTYPE_NIEUW`), using the old NWB numbering of `WVK_ID`
- `data/nwb_wegvakken_amsterdam.geojson`: current NWB road segments in Amsterdam (PDOK). From 2018 on, the crash data uses newer `WVK_ID`s that are only in this file.

**Steps**
1. Group the segments in `amsterdam_dertig.json` by road type
2. Match new NWB segments by location to the dertig segments
3. Group junctions by the segments that meet there
4. Add the group to the crashes

In [1]:
import json
import os
import urllib.request

import pandas as pd
import geopandas as gpd

In [2]:
CRASH_FILE = "ongevallen_gefilterd_amsterdam.csv"
DERTIG_FILE = "data/amsterdam_dertig.json"
NWB_FILE = "data/nwb_wegvakken_amsterdam.geojson"
NWB_URL = ("https://api.pdok.nl/rws/nationaal-wegenbestand-wegen/ogc/v1/collections/wegvakken/items"
           "?f=json&limit=1000&gme_naam=Amsterdam")
TRANSLATION_FILE = "data/road_id_translation.csv"
OUTPUT_FILE = "data/crashes_amsterdam_classified.csv"

BUFFER_M = 5          # max distance between old and new segment
MIN_OVERLAP = 0.5     # share of the new segment that must lie on old segments
MIN_PURITY = 0.8      # share of that overlap that must belong to one group

## 1. Group segments by road type
- **50->30**: GOW50 became GOW30 or ETW30 (changed)
- **stayed 50**: GOW50 stayed GOW50 (control group)
- **always 30**: was already ETW30 or GOW30
- **outside network**: no road type
- **other**: all other combinations

In [3]:
def road_group(old, new):
    """Group a dertig segment by its old and new road type."""
    if old in ("", "BUITEN NETWERK") and new in ("", "BUITEN N"):
        return "outside network"
    if old == "GOW50" and new in ("GOW30", "ETW30"):
        return "50->30"
    if old == new == "GOW50":
        return "stayed 50"
    if old == new and old in ("ETW30", "GOW30"):
        return "always 30"
    return "other"


# Dutch RD coordinates (EPSG:28992), so lengths are in metres
dertig = gpd.read_file(DERTIG_FILE).to_crs(28992)
dertig["WVK_ID"] = dertig["WVK_ID"].astype(str)
dertig["WEGTYPE_OUD"] = dertig["WEGTYPE_OUD"].fillna("")
dertig["WEGTYPE_NIEUW"] = dertig["WEGTYPE_NIEUW"].fillna("")
dertig["GROUP"] = [road_group(o, n) for o, n in zip(dertig["WEGTYPE_OUD"], dertig["WEGTYPE_NIEUW"])]
dertig = dertig.drop_duplicates("WVK_ID")

print(pd.crosstab([dertig["WEGTYPE_OUD"], dertig["WEGTYPE_NIEUW"]], dertig["GROUP"]))

GROUP                         50->30  always 30  other  outside network  \
WEGTYPE_OUD    WEGTYPE_NIEUW                                              
BUITEN NETWERK                     0          0      0             1511   
               ETW30               0          0     75                0   
               GOW30               0          0     13                0   
               GOW50               0          0      1                0   
ETW30                              0          0      3                0   
               ETW30               0      17001      0                0   
               GOW30               0          0     11                0   
ETW60          ETW30               0          0      1                0   
               ETW60               0          0    109                0   
               GOW50               0          0      1                0   
GOW30          GOW30               0        287      0                0   
GOW50          ETW30     

In [4]:
# Road length per group
(dertig.length.groupby(dertig["GROUP"]).sum() / 1000).round(0).rename("km")

GROUP
50->30              360.0
always 30          1371.0
other               303.0
outside network     272.0
stayed 50           277.0
Name: km, dtype: float64

In [5]:
# Segments with an ID from the dertig file
direct = pd.DataFrame({
    "ID": dertig["WVK_ID"],
    "ID_TYPE": "segment",
    "GROUP": dertig["GROUP"],
    "METHOD": "dertig_id",
    "DERTIG_WVK_ID": dertig["WVK_ID"],
})

## 2. Match new NWB segments by location
A new segment gets the group of the dertig segment it lies on: within 5 m, for at least half of its length, with at least 80% of that overlap in one group. Otherwise it is **unknown**.

The NWB file is downloaded once from PDOK and then cached.

In [6]:
def download_nwb():
    """Download all current NWB segments in Amsterdam, following the API's pages."""
    url = NWB_URL
    features = []
    while url:
        with urllib.request.urlopen(url, timeout=120) as response:
            page = json.load(response)
        features += page["features"]
        url = next((link["href"] for link in page["links"] if link["rel"] == "next"), None)
    with open(NWB_FILE, "w") as f:
        json.dump({"type": "FeatureCollection", "features": features}, f)
    print(f"Downloaded {len(features)} NWB segments to {NWB_FILE}")


if not os.path.exists(NWB_FILE):
    download_nwb()
nwb = gpd.read_file(NWB_FILE).to_crs(28992)
nwb["wvk_id"] = nwb["wvk_id"].astype(str)
nwb["length"] = nwb.length
print(f"{len(nwb)} NWB segments, {(~nwb['wvk_id'].isin(dertig['WVK_ID'])).sum()} of them not in the dertig file")

45870 NWB segments, 31055 of them not in the dertig file


In [7]:
buffered = dertig[["WVK_ID", "GROUP", "geometry"]].copy()
buffered["geometry"] = buffered.buffer(BUFFER_M)
new_only = nwb[~nwb["wvk_id"].isin(dertig["WVK_ID"])]

# Pieces of new segments that lie within the buffer of a dertig segment
pieces = gpd.overlay(new_only[["wvk_id", "length", "geometry"]], buffered,
                     how="intersection", keep_geom_type=False)
pieces["overlap"] = pieces.length

# Per new segment: the dertig segment with the most overlap, and how pure the group is
best = pieces.sort_values("overlap", ascending=False).drop_duplicates("wvk_id").set_index("wvk_id")
per_group = pieces.groupby(["wvk_id", "GROUP"])["overlap"].sum()
best["OVERLAP"] = (best["overlap"] / best["length"]).clip(upper=1)
best["PURITY"] = per_group.groupby(level=0).max() / per_group.groupby(level=0).sum()
confident = (best["OVERLAP"] >= MIN_OVERLAP) & (best["PURITY"] >= MIN_PURITY)

spatial = pd.DataFrame({
    "ID": best.index,
    "ID_TYPE": "segment",
    "GROUP": best["GROUP"].where(confident, "unknown").values,
    "METHOD": "nwb_location",
    "DERTIG_WVK_ID": best["WVK_ID"].values,
    "OVERLAP": best["OVERLAP"].round(3).values,
    "PURITY": best["PURITY"].round(3).values,
})
print(f"{confident.sum()} of {len(best)} new segments matched with confidence")
spatial["GROUP"].value_counts()

16194 of 27840 new segments matched with confidence


GROUP
unknown            11646
always 30           6954
50->30              5159
stayed 50           2440
outside network      959
other                682
Name: count, dtype: int64

## 3. Group junctions
A junction (`JTE_ID`) gets the group of the segments that start or end there (`jte_id_beg`/`jte_id_end` in the NWB). Residential streets (**always 30**) and **outside network** are ignored, so a junction on a main road takes the main road's group.
- Different groups left (e.g. 50->30 crossing stayed 50): **mixed**
- Only 30 streets and unmatched segments: **unknown**, since the unmatched segment could be a main road

In [8]:
SIDE_GROUPS = {"always 30", "outside network"}


def junction_group(groups):
    """Group a junction by the groups of the segments that meet there."""
    main = set(groups) - SIDE_GROUPS - {"unknown"}
    if len(main) == 1:
        return main.pop()
    if len(main) > 1:
        return "mixed"
    if "unknown" in groups:
        return "unknown"
    if "always 30" in groups:
        return "always 30"
    return "outside network"


# Group of every current NWB segment
segment_group = pd.concat([direct, spatial]).set_index("ID")["GROUP"]
nwb["GROUP"] = nwb["wvk_id"].map(segment_group).fillna("unknown")

ends = pd.concat([
    nwb[["jte_id_beg", "GROUP"]].rename(columns={"jte_id_beg": "jte_id"}),
    nwb[["jte_id_end", "GROUP"]].rename(columns={"jte_id_end": "jte_id"}),
]).dropna(subset=["jte_id"])
ends["jte_id"] = ends["jte_id"].astype("int64").astype(str)
connected = ends.groupby("jte_id")["GROUP"].agg(lambda g: sorted(set(g)))

junctions = pd.DataFrame({
    "ID": connected.index,
    "ID_TYPE": "junction",
    "GROUP": [junction_group(g) for g in connected],
    "METHOD": "junction",
    "CONNECTED_GROUPS": [", ".join(g) for g in connected],
})
junctions["GROUP"].value_counts()

GROUP
always 30          10380
unknown             8358
50->30              6408
stayed 50           2647
other                930
outside network      615
mixed                298
Name: count, dtype: int64

In [9]:
translation = pd.concat([direct, spatial, junctions], ignore_index=True)
translation.to_csv(TRANSLATION_FILE, index=False)
print(f"Saved {len(translation)} IDs to {TRANSLATION_FILE}")
translation.groupby(["ID_TYPE", "METHOD"])["GROUP"].value_counts().unstack(fill_value=0)

Saved 82762 IDs to data/road_id_translation.csv


GROUP                  50->30  always 30  mixed  other  outside network  \
ID_TYPE  METHOD                                                           
junction junction        6408      10380    298    930              615   
segment  dertig_id       3851      17288      0    841             1511   
         nwb_location    5159       6954      0    682              959   

GROUP                  stayed 50  unknown  
ID_TYPE  METHOD                            
junction junction           2647     8358  
segment  dertig_id          1795        0  
         nwb_location       2440    11646

## 4. Add the group to the crashes
- `LOCATION_TYPE`: segment or junction
- `ROAD_GROUP`: group of the road (or **unknown**)
- `MATCH_METHOD`: how the group was found (`dertig_id`, `nwb_location`, `junction` or `none`)
- `speed_changed`: True for 50->30, False for stayed 50, empty for the rest

In [10]:
crashes = pd.read_csv(CRASH_FILE, dtype=str, keep_default_na=False)
# IDs are stored as "263360031.0" in the file
for col in ["WVK_ID", "JTE_ID"]:
    crashes[col] = crashes[col].str.removesuffix(".0")

segments = translation[translation["ID_TYPE"] == "segment"].set_index("ID")
junction_table = translation[translation["ID_TYPE"] == "junction"].set_index("ID")

on_segment = crashes["WVK_ID"] != ""
crashes["LOCATION_TYPE"] = on_segment.map({True: "segment", False: "junction"})
crashes["ROAD_GROUP"] = crashes["WVK_ID"].map(segments["GROUP"]).where(
    on_segment, crashes["JTE_ID"].map(junction_table["GROUP"])).fillna("unknown")
crashes["MATCH_METHOD"] = crashes["WVK_ID"].map(segments["METHOD"]).where(
    on_segment, crashes["JTE_ID"].map(junction_table["METHOD"])).fillna("none")
crashes["speed_changed"] = crashes["ROAD_GROUP"].map({"50->30": "True", "stayed 50": "False"}).fillna("")

crashes.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(crashes)} crashes to {OUTPUT_FILE}")

Saved 63291 crashes to data/crashes_amsterdam_classified.csv


In [11]:
# Share of crashes with a known group, per year
((crashes["ROAD_GROUP"] != "unknown").groupby(crashes["JAAR_VKL"]).mean() * 100).round(0).rename("% known")

JAAR_VKL
2016    91.0
2017    92.0
2018    92.0
2019    88.0
2020    86.0
2021    86.0
2022    80.0
2023    85.0
2024    83.0
2025    84.0
Name: % known, dtype: float64

In [12]:
pd.crosstab(crashes["JAAR_VKL"], crashes["ROAD_GROUP"])

ROAD_GROUP,50->30,always 30,mixed,other,outside network,stayed 50,unknown
JAAR_VKL,,,,,,,
2016,2026,1986,61,1424,65,665,584
2017,2094,1794,85,1648,56,690,517
2018,1847,1779,88,2153,76,632,581
2019,1673,1516,88,2270,57,669,869
2020,1292,974,71,1416,56,487,722
2021,1427,1119,87,1331,53,548,752
2022,1449,1153,71,1415,59,610,1222
2023,1575,1416,73,1983,84,677,1005
2024,1399,1153,66,1838,91,607,1047


## 5. Validation: recorded speed limits
Each crash record has the speed limit the police recorded (`MAXSNELHD`). On roads that went from 50 to 30, it should go from mostly 50 before the change to mostly 30 after. On roads that stayed 50 it should stay 50.

The recorded speed is too noisy to classify roads with (e.g. a quarter of crashes on roads that stayed 50 are recorded as 30 after the change), but it is a good check of the classification above.

In [13]:
crashes["SPEED"] = pd.to_numeric(crashes["MAXSNELHD"], errors="coerce")
crashes["PERIOD"] = crashes["JAAR_VKL"].map(
    lambda y: "before (2021-2023)" if "2021" <= y <= "2023" else ("after (2024-2025)" if y >= "2024" else None))

# Share of crashes with a recorded speed of 30 or 50, per group and period
checked = crashes.dropna(subset=["PERIOD", "SPEED"])
checked = checked[checked["ROAD_GROUP"].isin(["50->30", "stayed 50", "always 30"])]
speed_share = pd.crosstab([checked["ROAD_GROUP"], checked["PERIOD"]], checked["SPEED"], normalize="index")[[30.0, 50.0]]
(speed_share * 100).round(0).rename(columns={30.0: "% recorded 30", 50.0: "% recorded 50"})

SPEED                          % recorded 30  % recorded 50
ROAD_GROUP PERIOD                                          
50->30     after (2024-2025)            87.0           12.0
           before (2021-2023)           13.0           86.0
always 30  after (2024-2025)            85.0           12.0
           before (2021-2023)           61.0           37.0
stayed 50  after (2024-2025)            25.0           71.0
           before (2021-2023)            5.0           90.0